<a href="https://colab.research.google.com/github/prithwis/parashar21/blob/main/K21_TwoPass.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![alt text](https://raw.githubusercontent.com/prithwis/parashar21/main/images/khona21-v0.png) <br>


---



[Prithwis Mukerjee](http://www.yantrajaal.com) ||
Website - [Parashar21](https://parashar21.blogspot.com)<br>

# Khona21 — AI-Assisted Jyotiṣa Interpretation

**Khona21** is an experimental AI-based Jyotiṣa interpretation platform, evolved from the original **Parashar21** project, which explored the application of artificial intelligence to traditional Indian astrology.

Named after the legendary Bengali astrologer Khona, the platform uses Large Language Models (LLMs) to interpret precomputed horoscopes against a curated corpus of Jyotiṣa literature, currently comprising works by **James Braha** and **Harihar Majumdar**.

Rather than relying on predefined interpretive rules, Khona21 uses source-grounded AI reasoning to identify planetary configurations, yogas, exceptions and conflicting interpretations, and synthesizes them into responses to specific questions.

This notebook, **k21_KonsultKhona**, demonstrates the platform's consultation workflow.

Khona21 is an experiment in applying modern AI to traditional knowledge systems, not a scientific validation of astrological predictions.

In [1]:
!pip -q install pypandoc
!pip -q install python-docx
from IPython.display import display, Markdown

!wget -O k21core.py -q https://raw.githubusercontent.com/prithwis/parashar21/main/k21utils/k21core.py
!wget -O k21fileops.py -q https://raw.githubusercontent.com/prithwis/parashar21/main/k21utils/k21fileops.py
from k21core import *
from k21fileops import *

In [ ]:
# ------------------------------------------------
# required for multiple imports of the same module
# ------------------------------------------------
#import k21core
#import importlib

#importlib.reload(k21core)
from k21core import *

In [2]:
# ------------------------------------------------
# Set Parameters
# ------------------------------------------------
provider = "openai"

#model = "gpt-5.6-luna"
model = "gpt-6-luna"
#model = "gpt-6-astra"

id = "TG"

###Access Control
This notebook accesses Google Drive to access files and uses the OpenAI API. Hence it can be used ONLY from the userid of the creator. Others will have to create their own access paths

In [3]:
# ------------------------------------------------
# Authenticate wit LLM provider using API keys stored as Colab Secret
# ------------------------------------------------
client = authenticateLLM(provider=provider)
# ------------------------------------------------
# Locate necessary file and set up paths to the same
# ------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')
#ls /content/drive/MyDrive/Prithwis/Parashar21/K21

K21_HOME = "/content/drive/MyDrive/Prithwis/Parashar21/K21"

# ------------------------------------------------
# The Corpus Files, Braha and Harihar, are available
# in GitHub Repo and can be downloaded with wget
# ------------------------------------------------
BRAHA   = f"{K21_HOME}/Corpus/BR_Curated_22.txt"
HARIHAR = f"{K21_HOME}/Corpus/HM_Clean_07.txt"
# ------------------------------------------------
# The Chart, Navamsha and Dasha data are generated by Parashar21
# Needs to be brought into this VM
# ------------------------------------------------
CHART   = f"{K21_HOME}/Natives/{id}-Chart.txt"
NAVAMSA = f"{K21_HOME}/Natives/{id}-Navamsa_Chart.txt"
DASHA   = f"{K21_HOME}/Natives/{id}-Dasha.txt"

OPENAI | User: Calcutta | Email: prithwis@yantrajaal.com | ID: user-sNAljKNmBf2igtAV2kUuqhoF
Mounted at /content/drive


In [4]:
# ------------------------------------------------
# Define the ROLE
# ------------------------------------------------
RoleFile = f"{K21_HOME}/Roles/khona261020.txt"
khona21 = readText(RoleFile)

/content/drive/MyDrive/Prithwis/Parashar21/K21/Roles/khona261020.txt | 3,404 characters


In [5]:
# ------------------------------------------------
# Set the Context
# ------------------------------------------------
BaseCONTEXT = buildContext(BRAHA, HARIHAR, CHART, NAVAMSA, DASHA)

/content/drive/MyDrive/Prithwis/Parashar21/K21/Corpus/BR_Curated_22.txt | 483,802 characters
/content/drive/MyDrive/Prithwis/Parashar21/K21/Corpus/HM_Clean_07.txt | 446,255 characters
/content/drive/MyDrive/Prithwis/Parashar21/K21/Natives/TG-Chart.txt | 4,864 characters
/content/drive/MyDrive/Prithwis/Parashar21/K21/Natives/TG-Navamsa_Chart.txt | 4,768 characters
/content/drive/MyDrive/Prithwis/Parashar21/K21/Natives/TG-Dasha.txt | 4,868 characters


In [6]:
# ------------------------------------------------
# FIRST PASS QUERY
#
# ------------------------------------------------
FIRST_PASS_QUERY = """
Prepare a comprehensive foundational assessment of this horoscope.

What are its principal strengths and weaknesses?

Identify the most significant favourable and unfavourable
planetary configurations, with explanations grounded in the
supplied reference texts.

First explain your findings in simple layman's language,
then justify them with astrological reasoning and specific
references to the supplied texts.

Pay particular attention to:
1. Significant yogas, planetary strengths and weaknesses,
   aspects, conjunctions and sign exchanges.
2. Contradictory indications, exceptions and mitigating factors.
3. The relative importance of different configurations
   in shaping the overall interpretation.

Distinguish clearly between chart facts and interpretations.
Do not invent configurations or apply rules unsupported by
the supplied reference texts.

This assessment will serve as the foundational reference
for subsequent questions about this horoscope. Therefore,
make it comprehensive, internally consistent and explicit
about uncertainties.

Do not make predictions about specific future events.
"""

In [7]:
# ------------------------------------------------
# Assemble the final query
# Call the LLM
# ------------------------------------------------

QUERY = buildQuery(BaseCONTEXT, FIRST_PASS_QUERY)

FirstPass = callLLM(
    client=client,
    role=khona21,
    query=QUERY,
    model=model,
    provider=provider,
    effort="medium"
)


OPENAI | gpt-6-luna | Effort: medium | In: 222,012 | Cached: 0 | Out: 17,724 | Reasoning: 11,018 | Time: 228.5s


In [ ]:
#display(Markdown(answer))

In [8]:
# ------------------------------------------------
# Save answer as markdown for machine use
# ------------------------------------------------
MD_FILE = saveAnswer(id, model, FirstPass)

Saved: /content/K21_TG_gpt-6-luna_261026_144317.md


In [9]:
def augmentContext(baseContext, firstPass):
    """
    Augment the original context with the FirstPass horoscope assessment.
    Returns a new context without modifying the original.
    """
    return (
        baseContext
        + "\n\n<FOUNDATIONAL_ASSESSMENT>\n"
        + firstPass
        + "\n</FOUNDATIONAL_ASSESSMENT>\n"
    )

In [10]:
context = augmentContext(BaseCONTEXT, FirstPass)

In [26]:
# ------------------------------------------------
# Ask your question / define the task
# This and subsequent cells can be run multiple times
# ------------------------------------------------
QUESTION = """
Use the original rashi chart, navamsha chart and the dasha information in the context, then add this additional gochar information given here.


HOROSCOPE DATA FOR JYOTISHA REASONING
=====================================

Analysis : Gochar
Chart Type : Rashi
Gender : x

HOUSE LORDSHIP
--------------
House 1 lord is Jupiter
House 2 lord is Saturn
House 3 lord is Saturn
House 4 lord is Jupiter
House 5 lord is Mars
House 6 lord is Venus
House 7 lord is Mercury
House 8 lord is Moon
House 9 lord is Sun
House 10 lord is Mercury
House 11 lord is Venus
House 12 lord is Mars

PLANETARY INFORMATION
---------------------

Lagna:
- Occupies Bhava 1
- Located in Rashi Sagittarius
- Located in Nakshatra 19 (Mula)
- Absolute celestial longitude 250.52 degrees

Sun:
- Occupies Bhava 11
- Located in Rashi Libra
- Located in Nakshatra 15 (Swati)
- Lord of Bhavas [9]
- Debilitated
- In enemy house
- Functional benefic
- Absolute celestial longitude 188.761 degrees

Moon:
- Occupies Bhava 5
- Located in Rashi Aries
- Located in Nakshatra 1 (Ashwini)
- Lord of Bhavas [8]
- Absolute celestial longitude 11.875 degrees

Mars:
- Occupies Bhava 8
- Located in Rashi Cancer
- Located in Nakshatra 9 (Ashlesha)
- Lord of Bhavas [5, 12]
- Debilitated
- In friendly house
- Functional benefic
- Absolute celestial longitude 111.524 degrees

Mercury:
- Occupies Bhava 11
- Located in Rashi Libra
- Located in Nakshatra 16 (Vishakha)
- Lord of Bhavas [7, 10]
- In friendly house
- Functional malefic
- Retrograde
- Absolute celestial longitude 206.44 degrees

Jupiter:
- Occupies Bhava 8
- Located in Rashi Cancer
- Located in Nakshatra 9 (Ashlesha)
- Lord of Bhavas [1, 4]
- Exalted
- In friendly house
- Absolute celestial longitude 119.376 degrees

Venus:
- Occupies Bhava 11
- Located in Rashi Libra
- Located in Nakshatra 14 (Chitra)
- Lord of Bhavas [6, 11]
- In own house
- In friendly house
- Functional malefic
- Retrograde
- Absolute celestial longitude 185.145 degrees

Saturn:
- Occupies Bhava 4
- Located in Rashi Pisces
- Located in Nakshatra 26 (UttaraBhadrapada)
- Lord of Bhavas [2, 3]
- Retrograde
- Absolute celestial longitude 345.423 degrees

Rahu:
- Occupies Bhava 3
- Located in Rashi Aquarius
- Located in Nakshatra 23 (Dhanishtha)
- Functional malefic
- Absolute celestial longitude 302.138 degrees

Ketu:
- Occupies Bhava 9
- Located in Rashi Leo
- Located in Nakshatra 10 (Magha)
- Functional malefic
- Absolute celestial longitude 122.138 degrees

PLANETARY ASPECTS
-----------------
Sun aspects ['Moon']
Moon aspects ['Sun', 'Venus', 'Mercury']
Mars aspects ['Sun', 'Venus', 'Mercury', 'Rahu']
Mercury aspects ['Moon']
Jupiter aspects ['Saturn']
Venus aspects ['Moon']
Saturn aspects ['Lagna']

PLANETS ASPECTED BY
--------------------
Moon is aspected by ['Sun', 'Venus', 'Mercury']
Sun is aspected by ['Moon', 'Mars']
Mercury is aspected by ['Moon', 'Mars']
Venus is aspected by ['Moon', 'Mars']
Rahu is aspected by ['Mars']
Saturn is aspected by ['Jupiter']
Lagna is aspected by ['Saturn']

PLANETARY CONJUNCTIONS
-----------------------
Sun conjuncts ['Venus', 'Mercury']
Mercury conjuncts ['Sun', 'Venus']
Venus conjuncts ['Sun', 'Mercury']
Mars conjuncts ['Jupiter']
Jupiter conjuncts ['Mars']

HOUSE LORD RELATIONSHIPS
-------------------------
Lord of Bhava 1 conjuncts ['Mars']
Lord of Bhava 4 conjuncts ['Mars']
Lord of Bhava 5 conjuncts ['Jupiter']
Lord of Bhava 6 conjuncts ['Sun', 'Mercury']
Lord of Bhava 7 conjuncts ['Sun', 'Venus']
Lord of Bhava 9 conjuncts ['Venus', 'Mercury']
Lord of Bhava 10 conjuncts ['Sun', 'Venus']
Lord of Bhava 11 conjuncts ['Sun', 'Mercury']
Lord of Bhava 12 conjuncts ['Jupiter']

LORD LORD RELATIONSHIPS
-----------------------
Lord of Bhava 1 connected with Lords [12, 5]
Lord of Bhava 4 connected with Lords [12, 5]
Lord of Bhava 5 connected with Lords [4, 1]
Lord of Bhava 6 connected with Lords [9, 10, 7]
Lord of Bhava 7 connected with Lords [9, 6, 11]
Lord of Bhava 9 connected with Lords [11, 10, 6, 7]
Lord of Bhava 10 connected with Lords [9, 6, 11]
Lord of Bhava 11 connected with Lords [9, 10, 7]
Lord of Bhava 12 connected with Lords [4, 1]

BHAVA ASPECTS
--------------
Bhava 1 receives aspect from ['Saturn']
Bhava 2 receives aspect from ['Mars', 'Jupiter']
Bhava 3 receives aspect from ['Mars', 'Ketu']
Bhava 4 receives aspect from ['Jupiter']
Bhava 5 receives aspect from ['Sun', 'Venus', 'Mercury']
Bhava 6 receives aspect from ['Saturn']
Bhava 7 receives aspect from ['Lagna']
Bhava 9 receives aspect from ['Rahu']
Bhava 10 receives aspect from ['Saturn']
Bhava 11 receives aspect from ['Moon', 'Mars']
Bhava 12 receives aspect from ['Jupiter']

BHAVA ASPECTED BY LORDS
------------------------
Bhava 1 receives aspect from Lords ['3', '2']
Bhava 2 receives aspect from Lords ['4', '5', '12', '1']
Bhava 3 receives aspect from Lords ['12', '5']
Bhava 4 receives aspect from Lords ['4', '1']
Bhava 5 receives aspect from Lords ['7', '10', '6', '11', '9']
Bhava 6 receives aspect from Lords ['3', '2']
Bhava 10 receives aspect from Lords ['3', '2']
Bhava 11 receives aspect from Lords ['12', '5', '8']
Bhava 12 receives aspect from Lords ['4', '1']

----------------------------------------------------
now using all the information including previously supplied dasha information tell us

Is there any possibility of the native moving houses? or changing residences in the near future.
"""

In [27]:
# ------------------------------------------------
# Assemble the query
# Call the LLM
# ------------------------------------------------

QUERY = buildQuery(context, QUESTION)

answer = callLLM(
    client=client,
    role=khona21,
    query=QUERY,
    model=model,
    provider=provider,
    effort="medium"
)

OPENAI | gpt-6-luna | Effort: medium | In: 230,111 | Cached: 0 | Out: 3,540 | Reasoning: 2,770 | Time: 43.7s


In [15]:
len(BaseCONTEXT), len(FirstPass), len(context)

(944827, 27909, 972792)

In [28]:
# ------------------------------------------------
# Save answer as markdown for machine use
# ------------------------------------------------
MD_FILE = saveAnswer(id, model, answer)

Saved: /content/K21_TG_gpt-6-luna_261026_153010.md


In [29]:
# ------------------------------------------------
# Convert markdown to docx for human readability
# Second Parameter is identifier of native, can be replaced by id
# ------------------------------------------------
DOCX_FILE = markdownToDocx(MD_FILE, "TopGun", QUESTION, provider,model)

Saved: /content/K21_TG_gpt-6-luna_261026_153010.docx


#ChronoTantra | ChronoYantra | ChronoMantra | ChronoSutra
 [More Information ](http://bit.ly/chronobooks) <br>

[![More Information](https://raw.githubusercontent.com/prithwis/parashar21/refs/heads/main/images/CTYMS-Colab-Footer.png)](https://chronos.yantrajaal.com)